# CMPE 460 TextCNN Training on Mendeley

**Member 2**

This notebook trains TextCNN on the prepared version 2 Mendeley baseline dataset. It reads the archive generated by `06_mendeley_data_preprocessing.ipynb`. Run that notebook first to create `data/mendeley/preprocessed.npz` in this checkout.

The architecture and initial training settings match the SecureAI-SE TextCNN experiment. The prepared archive supplies the Mendeley vocabulary, six label names, split assignments, and 512-character input length.

Training uses the training split. Validation macro-F1 selects the best epoch. Save the notebook with its outputs after training.


## 1. Load the Prepared Data

The archive records the source CSV checksum, input template, row assignments, vocabulary, and sequence length. This notebook loads its training and validation arrays. The file checksum is printed to identify the prepared input used by the experiment.

If imports fail, install dependencies once using `%pip install torch numpy pandas scikit-learn` in a separate code cell.


In [1]:
from time import perf_counter

setup_started = perf_counter()
print("Loading packages and prepared Mendeley data...", flush=True)

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_PATH = Path("data/mendeley/preprocessed.npz")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Run all cells in 06_mendeley_data_preprocessing.ipynb "
        f"to create {DATA_PATH.resolve()} in this checkout."
    )

with np.load(DATA_PATH, allow_pickle=False) as prepared:
    metadata = json.loads(prepared["metadata_json"].item())

    if (
        metadata.get("schema_version") != 1
        or metadata.get("dataset_id") != "zhzd5j9ffy"
        or metadata.get("dataset_version") != 2
    ):
        raise ValueError("The prepared file does not match the expected Mendeley preprocessing schema.")

    X = {
        split: prepared[f"X_{split}"]
        for split in ("train", "validation")
    }
    y = {
        split: prepared[f"y_{split}"]
        for split in ("train", "validation")
    }

MAX_LENGTH = metadata["max_length"]
PAD_ID = metadata["pad_id"]
UNK_ID = metadata["unk_id"]
char_to_id = metadata["char_to_id"]
label_names = metadata["label_names"]
DATASET_REVISION = metadata["dataset_doi"]

for split in ("train", "validation"):
    assert X[split].dtype == np.int64
    assert y[split].dtype == np.int64
    assert X[split].shape == (metadata["split_sizes"][split], MAX_LENGTH)
    assert y[split].shape == (metadata["split_sizes"][split],)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < metadata["vocabulary_size"]
    assert np.all((y[split] >= 0) & (y[split] < len(label_names)))
    print(f"{split}: X={X[split].shape}, y={y[split].shape}", flush=True)

prepared_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print(f"Prepared file SHA-256: {prepared_sha256}", flush=True)
print(f"Source CSV SHA-256: {metadata['source_csv_sha256']}", flush=True)
print(f"Data loaded in {perf_counter() - setup_started:.1f}s.", flush=True)
display(pd.Series(label_names, name="class").rename_axis("label_id").to_frame())


Loading packages and prepared Mendeley data...
train: X=(18015, 512), y=(18015,)
validation: X=(2573, 512), y=(2573,)
Prepared file SHA-256: 5308146e1ea6f365e4ac49dec8e3a43449a1d050f3b48abc1f44adba9477469e
Source CSV SHA-256: d3b3509555924de6e1d37522806f2debd97ea6659e8254a95ad98d19c28a24d6
Data loaded in 5.7s.


,class
label_id,
0,normal
1,xss
2,sqli
3,cmdi
4,lfi
5,ssrf


## 2. Configure Training and Create Data Loaders

Each batch contains up to 64 requests. Only the training loader shuffles the rows. The seed, PyTorch version, device, and CPU thread count are printed so that the experiment can be reproduced. The training loop reports progress every 20 batches.


In [2]:
SEED = 42
BATCH_SIZE = 64
EMBEDDING_DIM = 32
FILTERS_PER_KERNEL = 64
KERNEL_SIZES = (3, 4, 5)
DROPOUT = 0.3
LEARNING_RATE = 0.001
MAX_EPOCHS = 15
PATIENCE = 4
LOG_EVERY_BATCHES = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocabulary_size = len(char_to_id) + 2
number_of_classes = len(label_names)

loaders = {}
for split in ("train", "validation"):
    samples = TensorDataset(
        torch.from_numpy(X[split]),
        torch.from_numpy(y[split]),
    )
    loaders[split] = DataLoader(
        samples,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

print(f"Device: {device}", flush=True)
print(f"PyTorch version: {torch.__version__}", flush=True)
print(f"CPU threads: {torch.get_num_threads()}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
else:
    print("This kernel will train on the CPU.", flush=True)
print(f"Dataset revision: {DATASET_REVISION}", flush=True)
print(f"Vocabulary size: {vocabulary_size}", flush=True)
print(f"Number of classes: {number_of_classes}", flush=True)
print(f"Maximum input length: {MAX_LENGTH}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Learning rate: {LEARNING_RATE}", flush=True)
print(f"Maximum epochs: {MAX_EPOCHS}", flush=True)
print(f"Early-stopping patience: {PATIENCE}", flush=True)


Device: cpu
PyTorch version: 2.15.0+cpu
CPU threads: 12
This kernel will train on the CPU.
Dataset revision: 10.17632/zhzd5j9ffy.2
Vocabulary size: 222
Number of classes: 6
Maximum input length: 512
Batch size: 64
Learning rate: 0.001
Maximum epochs: 15
Early-stopping patience: 4


## 3. Define the TextCNN

The model uses 32-dimensional character embeddings, three convolutions with window sizes of 3, 4, and 5, and 64 filters per window size. ReLU and max pooling summarize the local patterns. Dropout of 0.3 precedes the six-class output layer.

Pooling uses windows containing real characters. Windows that include padding contribute zero after ReLU.

[PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html) · [PyTorch Conv1d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)


In [3]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        number_of_classes,
        embedding_dim=32,
        filters_per_kernel=64,
        kernel_sizes=(3, 4, 5),
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_id,
        )
        self.convolutions = nn.ModuleList([
            nn.Conv1d(embedding_dim, filters_per_kernel, kernel_size)
            for kernel_size in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            filters_per_kernel * len(kernel_sizes),
            number_of_classes,
        )

    def forward(self, tokens):
        embedded = self.embedding(tokens).transpose(1, 2)
        nonpadding = tokens.ne(self.pad_id)
        pooled_features = []

        for convolution in self.convolutions:
            features = torch.relu(convolution(embedded))
            kernel_size = convolution.kernel_size[0]
            valid_windows = nonpadding.unfold(1, kernel_size, 1).all(dim=-1)
            features = features.masked_fill(~valid_windows.unsqueeze(1), 0.0)
            pooled_features.append(features.amax(dim=2))

        combined = torch.cat(pooled_features, dim=1)
        return self.classifier(self.dropout(combined))


model = TextCNN(
    vocabulary_size=vocabulary_size,
    number_of_classes=number_of_classes,
    embedding_dim=EMBEDDING_DIM,
    filters_per_kernel=FILTERS_PER_KERNEL,
    kernel_sizes=KERNEL_SIZES,
    dropout=DROPOUT,
    pad_id=PAD_ID,
).to(device)

model.eval()
with torch.no_grad():
    example_inputs = torch.from_numpy(X["train"][:BATCH_SIZE]).to(device)
    example_scores = model(example_inputs)

assert example_scores.shape == (len(example_inputs), number_of_classes)
assert torch.isfinite(example_scores).all()

print(model)
print(f"Example output shape: {tuple(example_scores.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


TextCNN(
  (embedding): Embedding(222, 32, padding_idx=0)
  (convolutions): ModuleList(
    (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(32, 64, kernel_size=(4,), stride=(1,))
    (2): Conv1d(32, 64, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=192, out_features=6, bias=True)
)
Example output shape: (64, 6)
Trainable parameters: 33,030


## 4. Train and Select the Best Epoch

Cross-entropy compares the class scores with the integer labels. Training progress is printed after the first batch, every 20 batches, and the final batch of each epoch. The first batch may take longer while PyTorch initializes its execution kernels.

After each epoch, validation accuracy and macro-F1 are calculated. The best weights are kept in memory. Training stops after four consecutive epochs without an improvement in validation macro-F1. Epoch durations are recorded alongside the metrics.


In [4]:
from copy import deepcopy

from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        scores = model(inputs)
        loss = criterion(scores, targets)

        total_loss += loss.item() * targets.size(0)
        true_labels.extend(targets.cpu().tolist())
        predicted_labels.extend(scores.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            labels=list(range(number_of_classes)),
            average="macro",
            zero_division=0,
        ),
    }
    return metrics, true_labels, predicted_labels


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history_rows = []

print(f"Starting training for up to {MAX_EPOCHS} epochs on {device}.", flush=True)

training_started = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = perf_counter()
    model.train()
    total_training_loss = 0.0

    for batch_index, (inputs, targets) in enumerate(loaders["train"], start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(inputs)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        total_training_loss += loss.item() * targets.size(0)

        if (
            batch_index == 1
            or batch_index % LOG_EVERY_BATCHES == 0
            or batch_index == len(loaders["train"])
        ):
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
                f"batch {batch_index:02d}/{len(loaders['train'])} | "
                f"elapsed {perf_counter() - epoch_started:.1f}s",
                flush=True,
            )

    training_loss = total_training_loss / len(loaders["train"].dataset)
    validation, _, _ = evaluate_model(model, loaders["validation"])
    epoch_seconds = perf_counter() - epoch_started

    history_rows.append({
        "epoch": epoch,
        "train_loss": training_loss,
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["accuracy"],
        "validation_macro_f1": validation["macro_f1"],
        "epoch_seconds": epoch_seconds,
    })

    print(
        f"Epoch {epoch:02d} | train loss {training_loss:.4f} | "
        f"val loss {validation['loss']:.4f} | "
        f"val accuracy {validation['accuracy']:.4f} | "
        f"val macro-F1 {validation['macro_f1']:.4f} | "
        f"time {epoch_seconds:.1f}s",
        flush=True,
    )

    if validation["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation["macro_f1"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}.")
        break

model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(history_rows)

print(f"Restored the best epoch: {best_epoch}")
print(f"Total training time: {perf_counter() - training_started:.1f}s")
display(history.round(4))


Starting training for up to 15 epochs on cpu.
Epoch 01/15 | batch 01/282 | elapsed 0.1s
Epoch 01/15 | batch 20/282 | elapsed 1.3s
Epoch 01/15 | batch 40/282 | elapsed 2.5s
Epoch 01/15 | batch 60/282 | elapsed 3.6s
Epoch 01/15 | batch 80/282 | elapsed 4.8s
Epoch 01/15 | batch 100/282 | elapsed 5.8s
Epoch 01/15 | batch 120/282 | elapsed 6.9s
Epoch 01/15 | batch 140/282 | elapsed 8.0s
Epoch 01/15 | batch 160/282 | elapsed 8.9s
Epoch 01/15 | batch 180/282 | elapsed 9.9s
Epoch 01/15 | batch 200/282 | elapsed 10.9s
Epoch 01/15 | batch 220/282 | elapsed 11.8s
Epoch 01/15 | batch 240/282 | elapsed 12.8s
Epoch 01/15 | batch 260/282 | elapsed 13.8s
Epoch 01/15 | batch 280/282 | elapsed 14.7s
Epoch 01/15 | batch 282/282 | elapsed 14.8s
Epoch 01 | train loss 0.6722 | val loss 0.1238 | val accuracy 0.9685 | val macro-F1 0.9646 | time 15.3s
Epoch 02/15 | batch 01/282 | elapsed 0.1s
Epoch 02/15 | batch 20/282 | elapsed 0.9s
Epoch 02/15 | batch 40/282 | elapsed 1.9s
Epoch 02/15 | batch 60/282 | elapse

,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1,epoch_seconds
0,1,0.6722,0.1238,0.9685,0.9646,15.3392
1,2,0.2008,0.0516,0.9895,0.9870,14.5769
2,3,0.1220,0.0377,0.9899,0.9871,15.8584
3,4,0.0892,0.0321,0.9918,0.9902,16.7922
4,5,0.0654,0.0241,0.9949,0.9942,15.2414
5,6,0.0486,0.0250,0.9934,0.9921,15.0865
6,7,0.0394,0.0199,0.9953,0.9946,34.4639
7,8,0.0320,0.0232,0.9946,0.9934,42.6713
8,9,0.0263,0.0192,0.9957,0.9946,40.1471
9,10,0.0239,0.0183,0.9949,0.9942,38.7989


## 5. Report Validation Results

The following metrics describe the model selected using validation macro-F1. Record them as development results in the architecture comparison. Test metrics will be reported after the experiment settings have been selected.

Save this notebook with its outputs after the run completes. Run all cells again to start a new experiment from the recorded seed.


In [5]:
from sklearn.metrics import classification_report

validation_metrics, validation_labels, validation_predictions = evaluate_model(
    model,
    loaders["validation"],
)

print(f"Best epoch: {best_epoch}")
print(f"Validation accuracy: {validation_metrics['accuracy']:.4f}")
print(f"Validation macro-F1: {validation_metrics['macro_f1']:.4f}")
print()
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=list(range(number_of_classes)),
    target_names=label_names,
    digits=4,
    zero_division=0,
))


Best epoch: 9
Validation accuracy: 0.9957
Validation macro-F1: 0.9946

              precision    recall  f1-score   support

      normal     0.9975    0.9983    0.9979      1200
         xss     1.0000    0.9803    0.9901       305
        sqli     1.0000    1.0000    1.0000       321
        cmdi     0.9771    0.9961    0.9865       257
         lfi     0.9933    0.9933    0.9933       300
        ssrf     1.0000    1.0000    1.0000       190

    accuracy                         0.9957      2573
   macro avg     0.9947    0.9947    0.9946      2573
weighted avg     0.9958    0.9957    0.9957      2573

